# News To Stock Analyser 데이터준비

뉴스기사를 전달하면, 긍/부정분석 뿐아니라, 특정주식에 대한 긍/부정평가 처리 RAG 구현

In [1]:
%pip install -Uq datasets

Note: you may need to restart the kernel to use updated packages.


In [22]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = os.getenv('langsmith_key')
os.environ['LANGSMITH_PROJECT'] = 'skn23-langchain'
os.environ['OPENAI_API_KEY'] = os.getenv("openai_key")
os.environ['HF_TOKEN'] = os.getenv("HF_TOKEN")

## 데이터준비
https://huggingface.co/datasets/daekeun-ml/naver-news-summarization-ko

In [3]:
from datasets import load_dataset   # HuggingFace 데이터셋 로더

dataset = load_dataset('daekeun-ml/naver-news-summarization-ko')    # 데이터셋 다운로드
dataset

c:\Users\Playdata\nlp\nlp_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})

In [4]:
# 필터링 : train 데이터셋에서 category가 economy만 남김
economy_dataset = dataset['train'].filter(lambda row: row['category'] == 'economy')
print(len(economy_dataset))

17088


In [5]:
economy_dataset[10] # economy_dataset에서 11번쨰 샘플 반환

{'date': '2022-07-01 06:54:01',
 'category': 'economy',
 'press': 'SBS Biz ',
 'title': '글로벌 비즈 가트너 올해 전세계 스마트폰 판매량 7% 감소 전망',
 'document': '경제와이드 모닝벨 글로벌 비즈 임선우 외신캐스터 글로벌 비즈입니다. ◇ 올해 스마트폰 판매 감소 올해 전세계 스마트폰 판매량이 크게 줄어들 것이란 전망이 나왔습니다. 시장조사업체 가트너는 글로벌 스마트폰 판매가 7% 하락할 것으로 내다봤는데요. 경제 전반에 걸친 침체 우려와 중국의 봉쇄조치 여파 그리고 인플레이션으로 소비자들이 지갑을 열기 주저하면서 수요가 줄어들 것 이라고 설명했습니다. 그러면서 올해 전체 출하량은 14억6천만대 수준에 그칠 것으로 예측했는데요. 종전 전망치인 16억대에서 대폭 낮춰 잡았습니다. 특히 세계 최대 스마트폰 시장인 중국에서 판매량은 18%가 감소할 것으로 전망했는데요. 가트너는 이같은 수요 부진으로 애플을 비롯한 스마트폰 제조사부터 엔비디아 TSMC 같은 반도체 업체까지 압력이 가해질 것이라고 진단했습니다. ◇ EU 가상자산 돈세탁 막는다 유럽연합이 가상자산을 이용한 돈세탁을 막기위해 관련 기업을 규제하는 방안에 잠정 합의했습니다. 잠정안에는 가상자산 업체가 당국에 모든 디지털자산 거래에 대한 신원 확인 정보를 제공하도록 하는 내용이 담겼는데요. 이에 따라 업체들은 관련 개인정보를 확보해야하고 당국이 이를 요구할 경우 제출해야 합니다. 또 거래액이 1천 유로 우리돈 130만 원을 넘길 경우 비인증 거래소가 관리하는 가상자산 지갑도 똑같은 규칙이 적용되는데요. 여기에 더해 송금 규제를 활용해 거래를 상시 추적하고 불법성이 의심되는 거래를 막을 수 있도록 할 방침입니다. 이와 관련해 미국 최대 가상자산 거래소 코인베이스 등 관련 기업 40여 곳은 개인정보 침해 가능성을 언급하며 줄곧 반대 입장을 밝혀왔는데요. 하지만 최근 가상자산 관련 범죄 사례가 급증하고 있는 만큼 규제 움직임이 힘을

In [23]:
import pandas as pd

df = economy_dataset.to_pandas()    # Dataset을 DataFrame 형식으로 변환
df.head()

,date,category,press,title,document,link,summary
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.,https://n.news.naver.com/mnews/article/052/0001759333?sid=101,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, 정부가 하반기에 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 결정한 가운데, 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했다."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준비 7 8월 2만5000원 추가 시 와인 5종 및 생맥주 무제한 제공 인터컨티넨탈 서울 코엑스 브래서리 쿨 섬머 페스타 . 인터컨티넨탈 서울 코엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타 를 진행한다고 4일 밝혔다. 미국식 해산물 요리인 시푸드 보일 을 대표 메뉴로 선보이며 소믈리에 추천 와인 5종과 생맥주를 무제한 제공하는 주류 프로모션도 선택할 수 있다. 시푸드 보일 이 대표 메뉴로 준비되고 라이브 스테이션에서 셰프가 직접 원하는 메뉴를 먹기 좋게 잘라 제공한다. 시푸드 보일은 문어와 랍스터 대게 갑오징어 새우 소라 관자 낙지 등 해산물을 쪄낸 뒤 셰프의 비법 시즈닝으로 이국적인 감칠맛을 더한 메뉴다. 프로모션 기간에는 해물전 가리비 불도장 장어 데마끼 로제 해물 뇨끼 등 한식 중식 일식 양식 등 세계 각국의 해산물 메뉴도 즐길 수 있다. 소믈리에 추천 와인 5종과 생맥주를 무제한으로 제공하는 옵션도 선택할 수 있다. 제공되는 와인은 레드와 화이트 와인 각 2종 스파클링 와인 1종으로 취향에 따라 다양하게 즐길 수 있다. 해당 기간 동안 입구 와인셀렉션 코너에서 10만원 이상 와인 구매 시 호텔에서 제작한 주트백도 선물로 증정한다. 이용 가격은 이전과 동일하며 네이버 예약 시 10% 할인 혜택도 제공한다. 주류 무제한 혜택은 2만5000원 추가 시 이용할 수 있다.,https://n.news.naver.com/mnews/article/277/0005112302?sid=101,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타 를 진행하는데 미국식 해산물 요리인 시푸드 보일 을 대표 메뉴로 선보이며 소믈리에 추천 와인 5종과 생맥주를 무제한 제공하는 주류 프로모션도 선택할 수 있으며 프로모션 기간에는 해물전 가리비 불도장 장어 데마끼 로제 해물 뇨끼 등 한식 중식 일식 양식 등 세계 각국의 해산물 메뉴도 즐길 수 있다.
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사체 사업에 본격 진출한다고 1일 밝혔다. 에디슨이노는 전날 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이스홀딩스 대표이사를 사내이사로 선임했다. 이날 사업 목적에 위성체 발사와 우주선 위성시스템 등 항공 우주 분야와 자율 주행·그래핀 관련 사업을 사업목적에 추가했다. 사내이사에 이승영 카이스트 정밀기계공학 박사를 영입해 임플란트 관련 연구·생산기술을 보강했다. 서울대 화학생물공학 출신 최도영 씨도 선임해 현재 임상 시험계획 승인을 받은 생체흡수성 금속 리조멧 사업 확대를 도모한다 계획이다. 골절 수술 시 인체에 흡수되는 소재인 리조멧은 국내 임상 계획 승인과 중국 내 임상승인에 요구되는 시험을 통과한 상태다.,https://n.news.naver.com/mnews/article/003/0011279060?sid=101,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이스홀딩스 대표이사를 사내이사로 선임하고 위성체 발사와 우주선 위성시스템 등 항공 우주 분야와 자율 주행·그래핀 관련 사업을 사업목적에 추가하여 우주발사체 사업에 본격 진출한다고 1일 밝혔다.
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사이언스는 기존 해외사업개발실을 BD Business Development 1 3실로 확대 재편하고 글로벌 규제 및 허가 전담 조직인 Global RA Regulatory Affairs 실을 신설한다고 1일 밝혔다. SK바이오사이언스는 지난해 코로나19 COVID 19 백신 위탁 생산 등으로 주목받는 백신 기업으로 부상하며 글로벌 사업의 영역과 규모가 급속도로 성장 중이다. 이러한 성장 속도에 맞춰 기존 전담 조직인 해외사업개발실을 보다 세분화 및 전문화하고자 BD 1 3실로 확대 재편했다. BD 1 3실은 앞으로 기존에 영위 중인 백신 사업뿐만 아니라 세포·유전자치료제 CGT 등 신규 사업에 대한 △글로벌 네트워크들과의 공동 개발 △신규 C D MO 수주 △개발 제품 상업화 등 다양한 영역의 사업을 고도화하고 실행력을 높이는 업무를 담당한다. 또한 Global RA실을 신설해 미국 유럽 등 해외 선진국의 GMP Good Manufacturing Practice 를 확보하는 등 국제적인 수준의 관련 인증 및 허가 획득에도 더욱 박차를 가할 방침이다. CMC팀도 신설됐다. CMC는 화학 Chemistry 제조 Manufacturing 품질 Control 의 약자다. CMC팀은 완제 의약품을 만드는 공정 개발 process development 과 품질 관리 quality control 부문에서 핵심적인 역할을 수행한다. 연구부터 임상 허가 생산 품질에 이르는 GMP 관련 제반 업무를 관리한다. SK바이오사이언스는 이번 조직 개편이 글로벌 탑티어 바이오 기업으로의 성장을 더욱 앞당기고 초격차 경쟁력 확보의 계기가 될 것으로 기대하고 있다. SK바이오사이언스는 지난달 29일 국내 최초 코로나19 백신인 스카이코비원 SKYCovione 멀티주 의 품목허가를 획득했다. 이어 국가출하승인 및 WHO 등 해외 승인을 통해 국내외 백신 시장에 본격 진출할 예정이다.,https://n.n

In [7]:
df.head()

,date,category,press,title,document,link,summary
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ..."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...
4,2022-07-01 21:48:04,economy,경향신문,금융당국 “증시 변동성 완화 조치”,4일부터 석달간 증권사 신용융자담보비율 유지의무 면제 금융당국이 코스피지수가 장중 ...,https://n.news.naver.com/mnews/article/032/000...,1일 1일 금융위원회는 증권 유관기관과 금융시장합동점검회의를 열고 코스피지수가 장중...


## sLLM 답변데이터 생성
llm을 이용해서 sLLM이 답변했으면 하는 내용을 생성해낸다. 이때 답변을 품질이 중요하므로, 되도록 상위모델을 사용하는 것이 좋다.

In [8]:
# 금융뉴스 분석요 구조화 출력 스키마(Pydantic) + 프롬프트 템플릿 구성
from pydantic import BaseModel, Field   # 구조화 출력 스키마 정의
from typing import List, Optional
from langchain_core.prompts import ChatPromptTemplate   # 채팅 프롬프트 템플릿

class StockAnalysis(BaseModel):
    stock_related: bool = Field(description='뉴스와 주식 종목간의 연관성 여부')
    summary: str = Field(description='뉴스 요약')
    
    # 긍정적 영향 관련 변수 (default_factory=list : 인스턴스가 생성될 때마다 list() 호출해서 새 리스트를 만들어준다.)
    positive_stocks : List[str] = Field(description='긍정적 영향이 예상되는 주석 품목명 목록',default_factory=list)
    positive_keywords : List[str] = Field(description='긍정적 영향의 근거가 되는 키워드 목록',default_factory=list)
    positive_reasons : List[str] = Field(description='긍정적 영향이 예상되는 이유')
    
    # 부정적 영향 관련 변수
    negative_stocks : List[str] = Field(description='부정적 영향이 예상되는 주석 품목명 목록',default_factory=list)
    negative_keywords : List[str] = Field(description='부정적 영향의 근거가 되는 키워드 목록',default_factory=list)
    negative_reasons : List[str] = Field(description='부정적 영향이 예상되는 이유')
    

In [9]:
system_prompt = '''  # 모델 역할/출력 규칙을 고정하는 시스템 프롬프트
당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
'''

user_prompt = '''  # 분석 대상 뉴스 본문을 전달하는 사용자 프롬프트
다음 뉴스기사의 내용에 대해 심층적인 분석을 수행해주세요.

[news]
{news}
'''

prompt = ChatPromptTemplate.from_messages([
    ('system',system_prompt),
    ('human',user_prompt)
])

news = df['document'][1]
prompt.invoke({'news':news})

ChatPromptValue(messages=[SystemMessage(content="  # 모델 역할/출력 규칙을 고정하는 시스템 프롬프트\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n", additional_kwargs={}, response_metadata={}), HumanMessage(content='  # 분석 대상 뉴스 본문을 전달하는 사용자 프롬프트\n다음 뉴스기사의 내용에 대해 심층적인 분석을 수행해주세요.\n\n[news]\n문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준비 7 8월 2만5000원 추가 시 와인 5종 및 생맥주 무제한 제공 인터컨티넨탈 서울 코엑스 브래서리 쿨 섬머 페스타 . 인터컨티넨탈 서울 코엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타 를 진행한다고 4일 밝혔다. 미국식 해산물 요리인 시푸드 보일 을 대표 메뉴로 선보이며 소믈리에 추천 와인 5종과 생맥주를 무제한 제공하는 주류 프로모션

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model('openai:gpt-4.1-mini')

chain = prompt | llm.with_structured_output(StockAnalysis)  # 프롬프트 -> LLM -> StorckAnallysis 구조화 출력

def analyze_news(news):
    return chain.invoke({'news':news})

analyze_news(news)

StockAnalysis(stock_related=True, summary="인터컨티넨탈 서울 코엑스 호텔 1층 뷔페 레스토랑 브래서리가 6월 6일부터 8월 31일까지 해산물 중심 미국식 해물찜 '시푸드 보일'을 대표 메뉴로 한 '쿨 섬머 페스타' 프로모션을 진행한다. 다양한 해산물 요리와 주류(와인 5종, 생맥주) 무제한 옵션을 제공하며, 와인 구입 시 추가 사은품도 증정한다. 가격은 동일하고 예약시 할인 혜택이 있다.", positive_stocks=['삼원에프엔지', 'CJ프레시웨이', '신세계푸드', '호텔신라'], positive_keywords=['호텔 레스토랑 프로모션', '해산물 소비 확대', '와인 소비 증가', '프리미엄 외식 수요 증가'], positive_reasons=['해산물(문어, 랍스터, 대게, 새우 등) 소비 촉진으로 관련 수산물 가공/유통기업에 긍정적 영향을 줄 수 있음', '호텔/외식업체 이벤트 강화로 관련 기업 매출 증가 기대', '주류(와인, 생맥주) 수요 증가로 식음·주류 납품업체 수혜 가능'], negative_stocks=[], negative_keywords=[], negative_reasons=[])

In [11]:
news = df['document'][100]
display(news)
print()

analyze_news(news)

'해수부 5일 개정 공유수면 관리 및 매립에 관한 법률 시행 헤럴드경제 홍태화 기자 앞으로 공유수면관리청이 어업·환경 등에 영향을 미칠 것으로 예상되는 공유수면 점용·사용 허가를 할 때 미리 어업인 등 이해관계자들의 의견을 들어야 한다. 해양수산부는 5일 이같은 내용이 담긴 개정 공유수면 관리 및 매립에 관한 법률과 같은 법 시행령·시행규칙이 이날부터 시행된다고 밝혔다. 바다·바닷가·하천 등 공유수면은 공유재이기 때문에 이를 점용·사용하기 위해서는 별도의 허가를 받아야 한다. 최근 해상풍력 발전시설 해변을 이용한 관광시설 등 대규모 시설이 공유수면을 장기적으로 점용·사용하는 경우가 늘어났지만 이해 관계자의 의견을 사전에 수렴할 수 없는 문제가 있었다. 이에 공유수면 점용·사용으로 인한 사회적 갈등이 증가했다. 이러한 문제를 해결하기 위해 해수부는 지난 1월 공유수면 점용·사용 허가를 할 때 이해관계자의 의견을 듣도록 공유수면 관리 및 매립에 관한 법률을 개정했다. 법 개정에 따라 공유수면관리청이 해양환경·수산자원·자연경관 보호 등에 영향을 끼칠 수 있는 공유수면 점용·사용 신청을 받은 경우 이를 관보 공보 와 인터넷 홈페이지에 공고해야 한다. 또 점용·사용 허가를 했을 때 피해를 볼 것으로 예상되는 어업인에 대한 의견 조사도 별도로 진행해야 한다. 황준성 해수부 해양공간정책과장은 공유수면 점용·사용으로 인한 이해 관계자의 피해를 방지하려는 법령 개정의 취지를 달성할 수 있도록 각 공유수면관리청과 협력해 관련 제도의 차질 없는 운영을 지원하겠다 고 말했다.'

StockAnalysis(stock_related=True, summary='해양수산부가 7월 5일부터 공유수면 점용·사용 허가 시 어업인 등 이해관계자의 의견을 반드시 청취하도록 법률을 개정·시행한다. 최근 해상풍력발전소, 해변 관광시설 등 공유수면을 활용한 대규모 개발이 늘어났으나 이해관계자 갈등이 증가함에 따라, 앞으로 공유수면관리청이 영향을 미칠 수 있는 사업에 대해 사전 공고·의견수렴 절차를 거치게 됐다.', positive_stocks=[], positive_keywords=[], positive_reasons=[], negative_stocks=['씨에스윈드', '유니슨', 'SK에코플랜트', '동국S&C'], negative_keywords=['해상풍력 발전', '공유수면', '이해관계자 의견수렴', '허가절차 복잡화', '어업인 반발'], negative_reasons=['해상풍력발전 등 공유수면 사용기업 입장에서는 어업인 등 이해관계자의 의견수렴 및 반발 확대로 허가절차가 복잡·지연될 수 있음. 이는 해상풍력 관련 기업(씨에스윈드, 유니슨, SK에코플랜트, 동국S&C 등)의 사업 진행 속도를 늦추거나 비용 부담을 증가시키는 부정적 요인으로 작용할 수 있음.'])

In [13]:
df = df[:1000]  # 상위 1000개 샘플링
df['content'] = df['title'] + '\n' + df['document'] # 제목 + 본문으로 content 컬럼생성

pd.set_option('display.max_colwidth', None)         # 컬럼 내용이 잘리지 않도록 생성
df['content'].head()

0                                                                                                                                                                      추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 

In [14]:
# llm 질위 & 분석 데이터 1000개 생성
from tqdm.auto import tqdm  # 프로그래스바

results = []

for content in tqdm(df['content']):
    result = analyze_news(content)  # 분석 진행(StockAnysis 반환)
    results.append(result)          # 리스트에 추가
    
df['result'] = results  # rufrhk fltmxmfmf DF result 컬럼으로 추가
df.head()

 16%|█▋        | 163/1000 [19:06<1:38:05,  7.03s/it]


KeyboardInterrupt: 

In [28]:
# json 변환
# - pydantic.BaseModel.model_dump() -> dict
# - pydantic.BaseModel.model_dum_dswib() -> dict_str

# Pydantic 객체를 JSON 문자열로 변환하는 함수
def parse_to_json(obj):
    return obj.model_dump_json()    # StockAnalysis 객체 -> JSON 문자열로 파싱

df['result_json'] = df['result'].appy(parse_to_json)
df.head()

KeyError: 'result'

In [ ]:
df = df.dropna(subset =['result_json'])
df = df.reset_index(drop=True)
df.head()

## 학습용 데이터셋 변환
- system
- user(human)
- assistant(ai)

In [27]:

df['system'] = system_prompt

df = df.rename(columns={
    'content' : 'user',    
    'result_json' : 'assistant'
})

df[['system', 'user', 'assistant']]


KeyError: "['user', 'assistant'] not in index"

In [31]:
# DataFrame을 JSON 파일로 저장
df[['system','user','assistant']].to_json(
    'train_json',       # 저장 파일명(경로)
    orient = 'records',
    force_ascii=False,
    indent = 4          # 들여쓰기 4칸
)

import os
from datasets import Dataset

dataset = Dataset.from_pandas(df[['system','user','assistant']])
dataset.push_to_hub(
    'hyJung2247/naver-economy-news2stock'
)

KeyError: "['user', 'assistant'] not in index"

In [33]:
from datasets import load_dataset   # HuggingFace 데이터셋 로더

ds = load_dataset("capybaraOh/naver-economy-news2stock")    # 데이터셋 다운로드
ds

DatasetDict({
    train: Dataset({
        features: ['system', 'user', 'assistant'],
        num_rows: 1000
    })
})

In [35]:
ds.push_to_hub(
    'hyJung2247/naver-economy-news2stock',
    token= os.environ['HF_TOKEN']
)

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 56.51ba/s]
Processing Files (1 / 1): 100%|██████████| 1.91MB / 1.91MB, 1.37MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.56s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/hyJung2247/naver-economy-news2stock/commit/970fa6cc367133d3a657084e898c9d7cb6ed98c1', commit_message='Upload dataset', commit_description='', oid='970fa6cc367133d3a657084e898c9d7cb6ed98c1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/hyJung2247/naver-economy-news2stock', endpoint='https://huggingface.co', repo_type='dataset', repo_id='hyJung2247/naver-economy-news2stock'), pr_revision=None, pr_num=None)